# ĐỀ XUẤT (PROPOSAL)

## Dự đoán giá cổ phiếu VN-Index dựa trên biến động giá trong quá khứ

| | |
|---|---|
| **Môn học** | Học Máy (Machine Learning) |
| **Trường** | Đại học Sư phạm Kỹ thuật TP.HCM (HCMUTE) |
| **Sinh viên thực hiện** | Bá Hoài Sơn — Bùi Thanh Tú |
| **Mã thử nghiệm** | `VCB.VN` (Vietcombank — Yahoo Finance) |
| **Loại bài toán** | Hồi quy có giám sát trên chuỗi thời gian |

---

## 4.1. Giới thiệu về bài toán và dữ liệu

### 4.1.1. Bối cảnh

Thị trường chứng khoán Việt Nam được đo lường bởi chỉ số **VN-Index** — phản ánh xu hướng chung của các cổ phiếu niêm yết trên Sở Giao dịch Chứng khoán TP.HCM. Việc dự đoán biến động giá cổ phiếu trong tương lai dựa trên dữ liệu lịch sử có ý nghĩa lớn đối với:

- **Nhà đầu tư cá nhân và tổ chức**: hỗ trợ ra quyết định mua/bán, định lượng kỳ vọng lợi nhuận và rủi ro.
- **Quản trị danh mục**: tối ưu phân bổ tài sản, hedge rủi ro thị trường.
- **Nghiên cứu kinh tế**: kiểm chứng các giả thuyết về Hiệu Quả Thị Trường (Efficient Market Hypothesis — EMH).

Mặc dù theo dạng yếu của EMH, giá hiện tại đã phản ánh toàn bộ thông tin lịch sử (do đó dự đoán "chính xác" là không thể), trên thực tế vẫn tồn tại các *anomalies* và *patterns* ngắn hạn mà mô hình thống kê / học máy có thể khai thác. Đây là động lực để nhóm chọn đề tài này.

### 4.1.2. Câu hỏi nghiên cứu

> **Q1.** Dựa vào các đặc trưng kỹ thuật được tính từ giá lịch sử (lợi suất, đường trung bình động, RSI, độ biến động…), liệu có thể dự đoán **lợi suất** (return) của ngày giao dịch tiếp theo không?
>
> **Q2.** Khi suy ngược về miền giá, sai số tuyệt đối (theo VND) ở mức nào là có thể chấp nhận được?
>
> **Q3.** Mô hình tuyến tính (Linear Regression) có thực sự đủ, hay các mô hình phi tuyến (KNN, Random Forest) cho kết quả vượt trội?
>
> **Q4.** Voting Ensemble có giúp giảm phương sai và cải thiện hiệu năng so với từng mô hình đơn lẻ?

### 4.1.3. Tập dữ liệu

Nhóm thu thập dữ liệu giá lịch sử của mã **VCB** (Vietcombank — `VCB.VN` trên Yahoo Finance) qua thư viện [`yfinance`](https://pypi.org/project/yfinance/). Dữ liệu được lưu tại `data/vcb_stock.csv`.

**Lý do chọn VCB:**

- Vốn hóa lớn nhất nhì thị trường, chiếm tỷ trọng cao trong rổ VN-Index → đại diện tốt cho xu hướng thị trường ngân hàng.
- Thanh khoản cao → giá ít bị nhiễu do thao túng.
- Niêm yết từ 2009 → có lịch sử dài (>4.000 phiên ngày).

**Quy mô:** 7.673 quan sát (4.209 phiên ngày + 3.464 phiên giờ — khung 1h chỉ có cho 730 ngày gần nhất do giới hạn API).

**Các biến gốc (8 cột):** `Date`, `Open`, `High`, `Low`, `Close`, `Volume`, `Interval`, `Ticker`/`Symbol` — tổ hợp gồm biến **liên tục** (giá), **rời rạc** (volume), **phân loại** (Interval, Ticker), thỏa yêu cầu của môn học.

Sau khi sinh đặc trưng kỹ thuật (xem mục 4.2), tập đặc trưng mở rộng có **14 biến đầu vào** + **1 biến mục tiêu**.

In [1]:
import sys, pandas as pd
sys.path.insert(0, '../scripts')
from ml_utils import load_raw_data

df_raw = pd.read_csv('../data/vcb_stock.csv', parse_dates=['Date'])
print('Tổng số dòng:', len(df_raw))
print('Khung thời gian:')
print(df_raw.groupby('Interval').size().to_string())
print('Khoảng thời gian:', df_raw['Date'].min(), '→', df_raw['Date'].max())
df_raw.head()

Tổng số dòng: 7673
Khung thời gian:
Interval
1d    4209
1h    3464
Khoảng thời gian: 2009-06-30 00:00:00 → 2026-05-15 07:00:00+00:00


,Date,Close,High,Low,Open,Volume,Interval,Ticker,Symbol
0,2023-07-31 02:00:00+00:00,61404.683594,62341.136719,61270.902344,62207.359375,0,1h,VCB,VCB.VN
1,2023-07-31 03:00:00+00:00,61471.570312,61605.351562,61337.792969,61404.683594,128504,1h,VCB,VCB.VN
2,2023-07-31 04:00:00+00:00,61404.683594,61538.460938,61404.683594,61471.570312,54401,1h,VCB,VCB.VN
3,2023-07-31 06:00:00+00:00,61538.460938,61672.242188,61404.683594,61404.683594,327312,1h,VCB,VCB.VN
4,2023-07-31 07:00:00+00:00,61270.902344,62073.578125,61270.902344,61538.460938,0,1h,VCB,VCB.VN


## 4.2. Kế hoạch phân tích dữ liệu

### 4.2.1. Định nghĩa input/output

Một cạm bẫy nổi tiếng khi dự đoán giá cổ phiếu bằng học máy là **dự đoán trực tiếp giá tuyệt đối**. Do giá có xu hướng tăng theo thời gian, miền giá trị ở tập test thường nằm **ngoài** miền của tập train. Mô hình dựa trên cây (Random Forest) hay khoảng cách (KNN) **không thể ngoại suy** — sai số tăng vọt. Trong khi đó Linear Regression có vẻ "thắng" nhưng thực chất chỉ vì hệ số gần như sao chép giá hiện tại sang ngày mai.

Để các mô hình được đánh giá **công bằng**, nhóm phát biểu lại bài toán dưới dạng *stationary*:

$$\boxed{\;\hat r_{t+1} = f(\mathbf{x}_t),\quad \hat C_{t+1} = C_t \cdot (1 + \hat r_{t+1})\;}$$

trong đó $r_{t+1} = C_{t+1}/C_t - 1$ là **lợi suất** ngày kế tiếp.

| Thành phần | Biến |
|------------|------|
| **Output (Y)** | `Target_Return` — lợi suất kỳ kế tiếp |
| **Input (X) — 14 đặc trưng** | `Return_{1,2,3,5,10}`, `MA{5,10,20}_Ratio`, `Vol_{5,10}`, `RSI_14`, `HL_Range`, `OC_Range`, `Vol_Change` |

Chi tiết các đặc trưng:

- **Lợi suất quá khứ** `Return_k = Close_t / Close_{t-k} - 1` (k = 1, 2, 3, 5, 10): bắt động lượng (momentum).
- **Tỷ lệ với MA** `MA_w_Ratio = Close_t / MA_w - 1`: vị trí so với đường trung bình động (xu hướng).
- **Volatility** `Vol_w = std(Return_1) trong w phiên`: rủi ro rung lắc.
- **RSI(14)** — chỉ báo quá mua / quá bán cổ điển.
- **HL_Range** = `(High - Low)/Close`: biên độ trong ngày.
- **OC_Range** = `(Close - Open)/Open`: lực tăng/giảm trong ngày.
- **Vol_Change** — % thay đổi khối lượng giao dịch.

### 4.2.2. Độ đo và phương pháp đánh giá

Vì là bài toán hồi quy chuỗi thời gian, nhóm sử dụng **chia theo thứ tự thời gian** (chronological split, 80% train / 20% test, không shuffle) — phản ánh đúng kịch bản thực tế: huấn luyện trên quá khứ, dự đoán tương lai.

Bốn độ đo đánh giá:

| Metric | Đơn vị | Diễn giải |
|--------|--------|-----------|
| **MAE** | lợi suất / VND | Sai số tuyệt đối trung bình. |
| **RMSE** | lợi suất / VND | Phạt nặng các sai số lớn. |
| **R²** | — | Tỷ lệ phương sai được giải thích. |
| **DirAcc(%)** | % | Tỷ lệ dự đoán **đúng dấu** (lên/xuống) — quan trọng với chiến lược giao dịch. |

Kết quả sẽ được trình bày trên **hai miền**: lợi suất (đánh giá mô hình) và giá VND (trực giác cho người đọc).

### 4.2.3. Các phương pháp dự kiến

1. **Linear Regression** — đường cơ sở (baseline) tuyến tính có giải thích được.
2. **K-Nearest Neighbors (KNN)** — dự đoán theo các phiên "giống" trong quá khứ, không tham số, bắt mẫu cục bộ.
3. **Random Forest** — ensemble cây quyết định, mạnh trên đặc trưng dạng bảng, robust với nhiễu.
4. **Voting Ensemble** — trung bình dự đoán của 3 mô hình trên, kỳ vọng giảm phương sai và ổn định kết quả.

Tất cả mô hình đều được bao bằng `Pipeline(StandardScaler + estimator)` để công bằng giữa các thuật toán nhạy thang đo (KNN, LR) và các thuật toán dạng cây.

### 4.2.4. Kế hoạch thực hiện và phân công

| Tuần | Mục tiêu | Phụ trách chính |
|------|----------|-----------------|
| 1 | Tải dữ liệu qua `yfinance`, lưu CSV, EDA cơ bản | Bá Hoài Sơn |
| 2 | Feature engineering, kiểm tra correlation, train Linear Regression + KNN | Bá Hoài Sơn |
| 3 | Train Random Forest, Voting Ensemble, tinh chỉnh hyperparameter | Bùi Thanh Tú |
| 4 | So sánh, vẽ biểu đồ, viết Milestone | Cả nhóm |
| 5 | Hoàn thiện Report, chuẩn bị Presentation | Cả nhóm |

**Phân công cụ thể:**

| Thành viên | Đảm nhiệm |
|------------|-----------|
| **Bá Hoài Sơn** | Thu thập & làm sạch dữ liệu; EDA; Linear Regression; KNN; biểu đồ giá lịch sử & phân phối; Proposal; Presentation. |
| **Bùi Thanh Tú** | Feature engineering nâng cao (RSI, MA-ratio, volatility); Random Forest; Voting Ensemble; biểu đồ so sánh & dự đoán/thực tế; Milestone; Report. |

Cả hai cùng thảo luận về phương pháp, review code và viết phần "Hạn chế / Hướng phát triển" trong Report.

## 4.3. Kết quả kỳ vọng

- **MAPE giá** dưới 2 % (sai số trung bình ~1.5 nghìn VND trên giá ~80 nghìn VND).
- **DirAcc** > 50 % để có giá trị thực tiễn (dự đoán dấu tốt hơn random).
- Random Forest hoặc Voting Ensemble đứng đầu trên metric tổng hợp; Linear Regression đóng vai trò baseline.
- Nhận diện được hạn chế: mô hình **không** thể dự đoán các biến cố ngoại sinh (tin tức, sốc thị trường, chính sách) — sẽ được trao đổi kỹ trong Report.

---
*Kết thúc Đề xuất.*